# 09. Episode·첫 진입 기준 평가

세 번째 개선 항목만 적용한다. 모델, target, feature, scaling과 학습 결과는 바꾸지 않고 평가 단위를 매분 row에서 **positive episode와 첫 진입**으로 변경한다.

## 정의

- Positive episode: 같은 `run_id`에서 1분 간격으로 연속된 positive label 묶음
- Signal cluster: 같은 `run_id`에서 threshold 이상 점수가 1분 간격으로 연속된 묶음
- First entry: signal cluster의 첫 번째 row만 진입으로 계산
- First-entry precision: 첫 진입 row 중 target positive 비율
- Episode recall: positive episode 중 positive row에 first entry가 하나 이상 들어간 episode 비율

최종 threshold는 아직 고르지 않는다. 각 score의 OOF 상위 1·2.5·5·10·20%에 대응하는 **절대 score threshold**를 만들고 Test에도 같은 값을 적용한다. 다음 단계의 percentile/rolling calibration과 혼동하지 않도록 이 percentile은 평가 곡선의 x축으로만 사용한다.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENT.md').is_file() and (candidate / 'README.md').is_file():
            return candidate
    raise FileNotFoundError('프로젝트 루트를 찾지 못했습니다.')


PROJECT_ROOT = find_project_root()
PREPROCESS_ROOT = (PROJECT_ROOT / '../../results/preprocessing').resolve()
TRAINING_ROOT = (PROJECT_ROOT / '../../results/training').resolve()
RESULT_ROOT = (PROJECT_ROOT / '../../results/evaluation/episode_first_entry_v1').resolve()
RESULT_ROOT.mkdir(parents=True, exist_ok=True)

BASELINE_ROOT = TRAINING_ROOT / 'moderntcn_ohlc_60m_v1'
DOWNSIDE_ROOT = TRAINING_ROOT / 'moderntcn_downside_aware_tp3_v1'
UTILITY_ROOT = TRAINING_ROOT / 'moderntcn_direct_utility_lambda1_v1'

TOP_FRACTIONS = [0.01, 0.025, 0.05, 0.10, 0.20]
SCORE_COLUMNS = {
    'original_hard_probability': 'pred_open_hard_probability',
    'downside_aware_probability': 'pred_downside_aware_probability',
    'derived_mfe_minus_downside': 'pred_utility_3m',
    'direct_utility_lambda1': 'pred_direct_utility_lambda1',
}
TARGET_COLUMNS = {
    'original_tp3': 'target_tp3_3m',
    'downside_aware_tp3': 'target_downside_aware_tp3_3m',
}

print('result output:', RESULT_ROOT)

result output: /home/user/urbandatalab/YSLee/results/evaluation/episode_first_entry_v1


## 1. 세 실험의 OOF/Test 예측 정렬

모든 score가 같은 `sample_id`와 날짜를 가리키는지 검증한다. `run_id`는 positive와 signal의 연속성을 끊는 경계로 사용한다.

In [2]:
metadata = pd.read_parquet(
    PREPROCESS_ROOT / 'ohlc_60m_tp3pct_v1_metadata.parquet',
    columns=['sample_id', 'run_id', 'session', 'symbol', 'decision_timestamp'],
)


def load_phase(phase):
    baseline = pd.read_parquet(BASELINE_ROOT / f'{phase}_predictions.parquet')
    downside = pd.read_parquet(DOWNSIDE_ROOT / f'{phase}_predictions.parquet')
    utility = pd.read_parquet(UTILITY_ROOT / f'{phase}_predictions.parquet')

    frame = baseline.merge(
        downside[[
            'sample_id', 'target_downside_aware_tp3_3m',
            'pred_downside_aware_probability', 'both_sides_3pct',
        ]],
        on='sample_id', how='inner', validate='one_to_one',
    ).merge(
        utility[[
            'sample_id', 'excursion_utility_lambda1',
            'pred_direct_utility_lambda1',
        ]],
        on='sample_id', how='inner', validate='one_to_one',
    ).merge(
        metadata[['sample_id', 'run_id']],
        on='sample_id', how='left', validate='one_to_one',
    )
    frame['phase'] = phase.upper()
    frame = frame.sort_values(['run_id', 'decision_timestamp', 'sample_id']).reset_index(drop=True)
    assert frame['sample_id'].is_unique
    assert frame['run_id'].notna().all()
    assert frame[list(SCORE_COLUMNS.values())].notna().all().all()
    return frame


oof = load_phase('oof')
test = load_phase('test')
assert len(oof) == 31_081
assert len(test) == 11_097

display(pd.DataFrame([
    {
        'phase': name,
        'samples': len(frame),
        'sessions': frame['session'].nunique(),
        'symbols': frame['symbol'].nunique(),
        'runs': frame['run_id'].nunique(),
        'original_positive_rows': int(frame['target_tp3_3m'].sum()),
        'downside_aware_positive_rows': int(frame['target_downside_aware_tp3_3m'].sum()),
    }
    for name, frame in [('OOF', oof), ('Test', test)]
]))

,phase,samples,sessions,symbols,runs,original_positive_rows,downside_aware_positive_rows
0,OOF,31081,4,60,136,4294,3763
1,Test,11097,2,32,54,1057,980


## 2. Positive episode 생성

날짜나 종목이 같아도 원본 연속 구간인 `run_id`가 다르면 별개 episode다. 중간에 1분보다 큰 gap이 있으면 새 episode로 시작한다.

In [3]:
def add_episode_ids(frame, target_name, target_column):
    result = frame.copy()
    target = result[target_column].eq(1)
    previous_target = result.groupby('run_id', sort=False)[target_column].shift(fill_value=0).eq(1)
    previous_time = result.groupby('run_id', sort=False)['decision_timestamp'].shift()
    contiguous = result['decision_timestamp'].sub(previous_time).dt.total_seconds().eq(60)
    starts = target & ~(previous_target & contiguous)
    local_id = starts.groupby(result['run_id'], sort=False).cumsum()
    episode_column = f'{target_name}_episode_id'
    result[episode_column] = pd.Series(pd.NA, index=result.index, dtype='string')
    result.loc[target, episode_column] = (
        result.loc[target, 'run_id'].astype(str)
        + '::' + local_id.loc[target].astype(str)
    )
    return result, episode_column


def build_episode_table(frame, phase, target_name, target_column):
    with_ids, episode_column = add_episode_ids(frame, target_name, target_column)
    positive = with_ids[with_ids[target_column].eq(1)].copy()
    episodes = positive.groupby(episode_column, observed=True).agg(
        phase=('phase', 'first'),
        session=('session', 'first'),
        symbol=('symbol', 'first'),
        run_id=('run_id', 'first'),
        episode_start=('decision_timestamp', 'min'),
        episode_end=('decision_timestamp', 'max'),
        positive_rows=('sample_id', 'size'),
        max_mfe_3m=('mfe_3m', 'max'),
        max_downside_3m=('mae_3m', lambda x: float((-x).max())),
    ).reset_index().rename(columns={episode_column: 'episode_id'})
    episodes['target_name'] = target_name
    return with_ids, episode_column, episodes


phase_frames = {}
episode_tables = []
episode_columns = {}
for phase, base_frame in [('OOF', oof), ('Test', test)]:
    frame = base_frame
    for target_name, target_column in TARGET_COLUMNS.items():
        frame, episode_column, episodes = build_episode_table(
            frame, phase, target_name, target_column
        )
        episode_columns[target_name] = episode_column
        episode_tables.append(episodes)
    phase_frames[phase] = frame

positive_episodes = pd.concat(episode_tables, ignore_index=True)
episode_summary = positive_episodes.groupby(['phase', 'target_name']).agg(
    episodes=('episode_id', 'size'),
    positive_rows=('positive_rows', 'sum'),
    median_rows_per_episode=('positive_rows', 'median'),
    q90_rows_per_episode=('positive_rows', lambda x: x.quantile(0.9)),
    max_rows_per_episode=('positive_rows', 'max'),
).reset_index()
episode_summary['rows_per_episode'] = (
    episode_summary['positive_rows'] / episode_summary['episodes']
)
display(episode_summary)

,phase,target_name,episodes,positive_rows,median_rows_per_episode,q90_rows_per_episode,max_rows_per_episode,rows_per_episode
0,OOF,downside_aware_tp3,1713,3763,2.0,4.0,11,2.196731
1,OOF,original_tp3,1742,4294,2.0,5.0,15,2.464983
2,TEST,downside_aware_tp3,416,980,2.0,4.0,14,2.355769
3,TEST,original_tp3,418,1057,2.0,5.0,16,2.528708


## 3. OOF score threshold와 첫 진입 평가

각 score의 OOF 분포에서 threshold를 한 번 계산한다. OOF와 Test 모두 동일 절대값을 사용한다. Signal cluster가 positive episode보다 먼저 시작했다면 episode 안에 score가 계속 높더라도 추가 진입으로 세지 않는다.

In [4]:
threshold_rows = []
for score_name, score_column in SCORE_COLUMNS.items():
    for top_fraction in TOP_FRACTIONS:
        threshold_rows.append({
            'score_name': score_name,
            'score_column': score_column,
            'top_fraction_oof': top_fraction,
            'threshold': float(oof[score_column].quantile(1 - top_fraction)),
        })
thresholds = pd.DataFrame(threshold_rows)
display(thresholds)

,score_name,score_column,top_fraction_oof,threshold
0,original_hard_probability,pred_open_hard_probability,0.010,0.697774
1,original_hard_probability,pred_open_hard_probability,0.025,0.573678
2,original_hard_probability,pred_open_hard_probability,0.050,0.478163
3,original_hard_probability,pred_open_hard_probability,0.100,0.378919
4,original_hard_probability,pred_open_hard_probability,0.200,0.265880
5,downside_aware_probability,pred_downside_aware_probability,0.010,0.537527
6,downside_aware_probability,pred_downside_aware_probability,0.025,0.447460
7,downside_aware_probability,pred_downside_aware_probability,0.050,0.388154
8,downside_aware_probability,pred_downside_aware_probability,0.100,0.325950
9,downside_aware_probability,pred_downside_aware_probability,0.200,0.246533


In [5]:
def evaluate_first_entries(
    frame,
    phase,
    score_name,
    score_column,
    target_name,
    target_column,
    episode_column,
    top_fraction,
    threshold,
):
    selected = frame[score_column].ge(threshold)
    previous_selected = selected.groupby(frame['run_id'], sort=False).shift(fill_value=False)
    previous_time = frame.groupby('run_id', sort=False)['decision_timestamp'].shift()
    contiguous = frame['decision_timestamp'].sub(previous_time).dt.total_seconds().eq(60)
    signal_start = selected & ~(previous_selected & contiguous)

    first = frame.loc[signal_start, [
        'sample_id', 'session', 'symbol', 'run_id', 'decision_timestamp',
        target_column, episode_column, score_column,
    ]].copy()
    first = first.rename(columns={
        target_column: 'is_positive',
        episode_column: 'episode_id',
        score_column: 'score',
    })
    first['phase'] = phase
    first['score_name'] = score_name
    first['target_name'] = target_name
    first['top_fraction_oof'] = top_fraction
    first['threshold'] = threshold

    total_episodes = int(frame[episode_column].dropna().nunique())
    captured_episodes = int(
        first.loc[first['is_positive'].eq(1), 'episode_id'].dropna().nunique()
    )
    selected_positive_rows = int(
        (selected & frame[target_column].eq(1)).sum()
    )
    positive_rows = int(frame[target_column].sum())

    result = {
        'phase': phase,
        'score_name': score_name,
        'target_name': target_name,
        'top_fraction_oof': top_fraction,
        'threshold': threshold,
        'samples': len(frame),
        'selected_rows': int(selected.sum()),
        'selected_row_rate': float(selected.mean()),
        'signal_clusters': len(first),
        'selected_rows_per_cluster': float(selected.sum() / max(len(first), 1)),
        'row_precision': float(
            frame.loc[selected, target_column].mean() if selected.any() else np.nan
        ),
        'row_recall': float(selected_positive_rows / max(positive_rows, 1)),
        'first_entry_precision': float(
            first['is_positive'].mean() if len(first) else np.nan
        ),
        'positive_episodes': total_episodes,
        'captured_episodes': captured_episodes,
        'episode_recall': float(captured_episodes / max(total_episodes, 1)),
        'entries_per_1000_rows': float(len(first) / len(frame) * 1000),
    }
    return result, first


metric_rows = []
daily_rows = []
first_entry_frames = []

for threshold_row in thresholds.itertuples(index=False):
    for phase, frame in phase_frames.items():
        for target_name, target_column in TARGET_COLUMNS.items():
            metric, first = evaluate_first_entries(
                frame=frame,
                phase=phase,
                score_name=threshold_row.score_name,
                score_column=threshold_row.score_column,
                target_name=target_name,
                target_column=target_column,
                episode_column=episode_columns[target_name],
                top_fraction=threshold_row.top_fraction_oof,
                threshold=threshold_row.threshold,
            )
            metric_rows.append(metric)
            first_entry_frames.append(first)

            for session, session_frame in frame.groupby('session', sort=False):
                daily_metric, _ = evaluate_first_entries(
                    frame=session_frame.reset_index(drop=True),
                    phase=phase,
                    score_name=threshold_row.score_name,
                    score_column=threshold_row.score_column,
                    target_name=target_name,
                    target_column=target_column,
                    episode_column=episode_columns[target_name],
                    top_fraction=threshold_row.top_fraction_oof,
                    threshold=threshold_row.threshold,
                )
                daily_metric['session'] = session
                daily_rows.append(daily_metric)

episode_metrics = pd.DataFrame(metric_rows)
daily_episode_metrics = pd.DataFrame(daily_rows)
first_entries = pd.concat(first_entry_frames, ignore_index=True)

print('metric rows:', len(episode_metrics))
print('first-entry rows:', len(first_entries))

metric rows: 80
first-entry rows: 53782


## 4. OOF 상위 5% 기준 핵심 비교

Row precision과 first-entry precision의 차이가 중복 신호 효과다. Episode recall은 positive row recall보다 실제 기회 포착에 가까운 값이다.

In [6]:
focus = episode_metrics[
    episode_metrics['top_fraction_oof'].eq(0.05)
].copy()
focus['precision_drop_from_dedup'] = (
    focus['first_entry_precision'] - focus['row_precision']
)
focus_view = focus[[
    'phase', 'target_name', 'score_name', 'threshold',
    'selected_row_rate', 'signal_clusters',
    'row_precision', 'first_entry_precision', 'precision_drop_from_dedup',
    'row_recall', 'episode_recall', 'entries_per_1000_rows',
]]
display(focus_view.sort_values(['target_name', 'score_name', 'phase']))

,phase,target_name,score_name,threshold,selected_row_rate,signal_clusters,row_precision,first_entry_precision,precision_drop_from_dedup,row_recall,episode_recall,entries_per_1000_rows
49,OOF,downside_aware_tp3,derived_mfe_minus_downside,0.005552,0.050031,1133,0.259807,0.235658,-0.024150,0.107361,0.154116,36.453139
51,Test,downside_aware_tp3,derived_mfe_minus_downside,0.005552,0.021988,199,0.200820,0.201005,0.000185,0.050000,0.096154,17.932775
69,OOF,downside_aware_tp3,direct_utility_lambda1,0.006105,0.050031,1107,0.247588,0.214995,-0.032593,0.102312,0.137186,35.616615
71,Test,downside_aware_tp3,direct_utility_lambda1,0.006105,0.026043,240,0.186851,0.170833,-0.016018,0.055102,0.096154,21.627467
29,OOF,downside_aware_tp3,downside_aware_probability,0.388154,0.050159,750,0.339962,0.344000,0.004038,0.140845,0.136602,24.130498
31,Test,downside_aware_tp3,downside_aware_probability,0.388154,0.013157,89,0.321918,0.337079,0.015161,0.047959,0.069712,8.020186
9,OOF,downside_aware_tp3,original_hard_probability,0.478163,0.050031,602,0.349839,0.352159,0.002320,0.144566,0.115003,19.368746
11,Test,downside_aware_tp3,original_hard_probability,0.478163,0.018203,86,0.316832,0.337209,0.020378,0.065306,0.060096,7.749842
48,OOF,original_tp3,derived_mfe_minus_downside,0.005552,0.050031,1133,0.297106,0.276258,-0.020848,0.107592,0.173364,36.453139
50,Test,original_tp3,derived_mfe_minus_downside,0.005552,0.021988,199,0.245902,0.251256,0.005355,0.056764,0.119617,17.932775


## 5. 전체 threshold 곡선과 날짜 안정성

Test의 실제 선택 비율이 OOF 목표 비율과 얼마나 달라지는지도 함께 본다. Rolling calibration은 다음 노트북에서 이 차이를 줄이는 방식으로 설계한다.

In [7]:
original_curve = episode_metrics[
    episode_metrics['target_name'].eq('original_tp3')
][[
    'phase', 'score_name', 'top_fraction_oof', 'threshold',
    'selected_row_rate', 'signal_clusters', 'first_entry_precision',
    'episode_recall', 'entries_per_1000_rows',
]].sort_values(['score_name', 'top_fraction_oof', 'phase'])
display(original_curve)

daily_focus = daily_episode_metrics[
    daily_episode_metrics['top_fraction_oof'].eq(0.05)
    & daily_episode_metrics['target_name'].eq('original_tp3')
][[
    'phase', 'session', 'score_name', 'signal_clusters',
    'first_entry_precision', 'episode_recall', 'entries_per_1000_rows',
]].sort_values(['session', 'score_name'])
display(daily_focus)

,phase,score_name,top_fraction_oof,threshold,selected_row_rate,signal_clusters,first_entry_precision,episode_recall,entries_per_1000_rows
40,OOF,derived_mfe_minus_downside,0.010,0.010829,0.010006,257,0.455253,0.066016,8.268717
42,Test,derived_mfe_minus_downside,0.010,0.010829,0.001983,20,0.550000,0.026316,1.802289
44,OOF,derived_mfe_minus_downside,0.025,0.007622,0.025031,598,0.334448,0.112514,19.240050
46,Test,derived_mfe_minus_downside,0.025,0.007622,0.008741,82,0.426829,0.083732,7.389385
48,OOF,derived_mfe_minus_downside,0.050,0.005552,0.050031,1133,0.276258,0.173364,36.453139
50,Test,derived_mfe_minus_downside,0.050,0.005552,0.021988,199,0.251256,0.119617,17.932775
52,OOF,derived_mfe_minus_downside,0.100,0.003735,0.100029,2040,0.208333,0.235936,65.634954
54,Test,derived_mfe_minus_downside,0.100,0.003735,0.054609,450,0.204444,0.210526,40.551500
56,OOF,derived_mfe_minus_downside,0.200,0.002053,0.200026,3555,0.167651,0.323766,114.378559
58,Test,derived_mfe_minus_downside,0.200,0.002053,0.133189,925,0.143784,0.306220,83.355862


,phase,session,score_name,signal_clusters,first_entry_precision,episode_recall,entries_per_1000_rows
147,OOF,session_2026-07-10,derived_mfe_minus_downside,348,0.272989,0.156830,38.338658
207,OOF,session_2026-07-10,direct_utility_lambda1,369,0.233062,0.141653,40.652198
87,OOF,session_2026-07-10,downside_aware_probability,317,0.444795,0.195616,34.923433
27,OOF,session_2026-07-10,original_hard_probability,232,0.482759,0.161889,25.559105
144,OOF,session_2026-07-13,derived_mfe_minus_downside,284,0.281690,0.195251,38.920104
204,OOF,session_2026-07-13,direct_utility_lambda1,189,0.227513,0.113456,25.901055
84,OOF,session_2026-07-13,downside_aware_probability,119,0.579832,0.145119,16.308072
24,OOF,session_2026-07-13,original_hard_probability,113,0.513274,0.131926,15.485816
146,OOF,session_2026-07-14,derived_mfe_minus_downside,235,0.242553,0.193772,38.398693
206,OOF,session_2026-07-14,direct_utility_lambda1,227,0.237885,0.183391,37.091503


## 6. Artifact 저장

In [8]:
artifact_paths = {
    'thresholds': RESULT_ROOT / 'oof_score_thresholds.parquet',
    'episode_metrics': RESULT_ROOT / 'episode_metrics.parquet',
    'daily_episode_metrics': RESULT_ROOT / 'daily_episode_metrics.parquet',
    'first_entries': RESULT_ROOT / 'first_entries.parquet',
    'positive_episodes': RESULT_ROOT / 'positive_episodes.parquet',
    'manifest': RESULT_ROOT / 'manifest.json',
}
thresholds.to_parquet(artifact_paths['thresholds'], index=False, compression='zstd')
episode_metrics.to_parquet(artifact_paths['episode_metrics'], index=False, compression='zstd')
daily_episode_metrics.to_parquet(
    artifact_paths['daily_episode_metrics'], index=False, compression='zstd'
)
first_entries.to_parquet(artifact_paths['first_entries'], index=False, compression='zstd')
positive_episodes.to_parquet(
    artifact_paths['positive_episodes'], index=False, compression='zstd'
)

manifest = {
    'experiment': 'episode_first_entry_v1',
    'created_by_notebook': 'notebooks/09_episode_first_entry_evaluation.ipynb',
    'positive_episode_rule': 'same run_id, target=1, consecutive 1-minute rows',
    'signal_cluster_rule': 'same run_id, score>=threshold, consecutive 1-minute rows',
    'entry_rule': 'first row of each signal cluster only',
    'episode_recall_rule': 'episode contains at least one positive first-entry row',
    'top_fractions_oof': TOP_FRACTIONS,
    'threshold_rule': 'absolute threshold calculated from pooled OOF score distribution and reused on Test',
    'threshold_role': 'diagnostic curve only; not final selection',
    'score_columns': SCORE_COLUMNS,
    'target_columns': TARGET_COLUMNS,
    'test_is_pristine': False,
    'artifact_paths': {key: str(value) for key, value in artifact_paths.items()},
}
artifact_paths['manifest'].write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8'
)

display(pd.DataFrame([
    {'artifact': name, 'path': str(path), 'size_mb': path.stat().st_size / 1024**2}
    for name, path in artifact_paths.items()
]))
print('저장 완료')

,artifact,path,size_mb
0,thresholds,/home/user/urbandatalab/YSLee/results/evaluati...,0.003266
1,episode_metrics,/home/user/urbandatalab/YSLee/results/evaluati...,0.015342
2,daily_episode_metrics,/home/user/urbandatalab/YSLee/results/evaluati...,0.023024
3,first_entries,/home/user/urbandatalab/YSLee/results/evaluati...,0.362783
4,positive_episodes,/home/user/urbandatalab/YSLee/results/evaluati...,0.080976
5,manifest,/home/user/urbandatalab/YSLee/results/evaluati...,0.001765


저장 완료


## 7. 판정

이번 단계에서는 모델 성능을 새로 만든 것이 아니라, 중복된 row 지표가 실제 첫 진입에서 얼마나 낮아지는지 측정한다. 다음 단계에서는 여기서 확인한 첫 진입 지표를 기준으로 OOF percentile과 날짜 순서 rolling calibration을 비교한다.

In [9]:
summary = focus[
    focus['target_name'].eq('original_tp3')
][[
    'phase', 'score_name', 'row_precision', 'first_entry_precision',
    'row_recall', 'episode_recall', 'selected_row_rate', 'signal_clusters',
]].sort_values(['score_name', 'phase'])
display(summary)

best_oof_precision = focus[
    focus['phase'].eq('OOF') & focus['target_name'].eq('original_tp3')
].sort_values('first_entry_precision', ascending=False).iloc[0]
best_test_precision = focus[
    focus['phase'].eq('Test') & focus['target_name'].eq('original_tp3')
].sort_values('first_entry_precision', ascending=False).iloc[0]
print('OOF top first-entry precision score:', best_oof_precision['score_name'])
print('Test top first-entry precision score:', best_test_precision['score_name'])
print('다음: 최종 score를 정하지 않고 각 score에 percentile/rolling calibration을 적용해 안정성을 비교합니다.')

,phase,score_name,row_precision,first_entry_precision,row_recall,episode_recall,selected_row_rate,signal_clusters
48,OOF,derived_mfe_minus_downside,0.297106,0.276258,0.107592,0.173364,0.050031,1133
50,Test,derived_mfe_minus_downside,0.245902,0.251256,0.056764,0.119617,0.021988,199
68,OOF,direct_utility_lambda1,0.270740,0.237579,0.098044,0.148680,0.050031,1107
70,Test,direct_utility_lambda1,0.214533,0.195833,0.058657,0.110048,0.026043,240
28,OOF,downside_aware_probability,0.468890,0.440000,0.170238,0.156716,0.050159,750
30,Test,downside_aware_probability,0.452055,0.438202,0.062441,0.083732,0.013157,89
8,OOF,original_hard_probability,0.515113,0.460133,0.186539,0.138921,0.050031,602
10,Test,original_hard_probability,0.460396,0.430233,0.087985,0.071770,0.018203,86


OOF top first-entry precision score: original_hard_probability
Test top first-entry precision score: downside_aware_probability
다음: 최종 score를 정하지 않고 각 score에 percentile/rolling calibration을 적용해 안정성을 비교합니다.
